# 04_sensor_silver

**Purpose:** Merge the PI-style simulated sensor history and live feed into one clean table (DEC-034). All readings are
simulated; this is not AVEVA PI System.

**Input tables:** `bronze_sensor_backfill` (2025-11 hourly history), `bronze_sensor_readings` (Eventstream feed, with
`ts_utc` and `sent_utc` as strings), `sensor_asset_hierarchy`; `Files/config/sensor.json`; the backfill truth file and
the stream run manifests in `Files/raw/sensor/`. Names are in the `layers` section of the config.

**Output tables:**
- `silver_sensor_readings` (overwrite): one row per tag and timestamp, with `quality` (Good / Questionable / Bad),
  `quality_reason`, `is_late` and `is_received`. Expected readings that never arrived are kept as Bad rows with no value.
- `sensor_silver_rejects` (append, keyed by `run_id`): duplicates, null keys and unknown tags, with the reason
- `sensor_quality_test_results` (append): the rule tests of this run
- `pipeline_run_log` (append): one row, written only after every test passes; 05_sensor_gold refuses a run without it

**Rules:** the same as `sensor/quality.py`. Rule tests use the 2025-11 history, the only data with deliberate bad readings:
the counts must equal the injections in the truth file. The live feed has no bad readings; for it, the tests check that
all readings are Good and that duplicates and late readings match the run manifests.

**Run order:** step 1 of 2 in `pl_sensor_hourly`: **04_sensor_silver** -> 05_sensor_gold.


In [ ]:
import glob, json, uuid
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")
RUN_ID = str(uuid.uuid4())[:8]
with open("/lakehouse/default/Files/config/sensor.json") as fh:
    CFG = json.load(fh)
L = CFG["layers"]
Q = CFG["quality"]
STREAM = CFG["stream"]
started = datetime.now()
print("RUN_ID:", RUN_ID)

# Both sources in one schema: the stream arrives with string timestamps and has a run ID and a send time
backfill = spark.table(L["bronze_backfill_table"]).select(
    "tag_name", F.col("ts_utc").cast("timestamp").alias("ts_utc"), F.lit(None).cast("timestamp").alias("sent_utc"),
    F.col("value").cast("double").alias("value"), "source", F.lit(None).cast("string").alias("stream_run_id"))
stream = spark.table(L["bronze_stream_table"]).select(
    "tag_name", F.col("ts_utc").cast("timestamp").alias("ts_utc"), F.col("sent_utc").cast("timestamp").alias("sent_utc"),
    F.col("value").cast("double").alias("value"), "source", F.col("run_id").alias("stream_run_id"))
received = backfill.unionByName(stream).cache()
rows_in = received.count()

tags = spark.table(L["hierarchy_table"]).filter(F.col("node_type") == "tag").select(
    "tag_name", "well_key", "facility_id", "product", "tag_code", "unit",
    "physical_min", "physical_max", "normal_min", "normal_max")
frozen_checked = spark.createDataFrame([(t["code"], t["frozen_check"]) for t in CFG["tags"]],
                                       "tag_code string, frozen_check boolean")
print("rows in Bronze:", rows_in)


In [ ]:
# Quarantine: null keys, unknown tags and every copy of a key but the first to arrive
first_arrival = Window.partitionBy("tag_name", "ts_utc").orderBy(F.col("sent_utc").asc_nulls_first())
checked = (received.join(tags.select("tag_name", F.lit(True).alias("_known_tag")), "tag_name", "left")
    .withColumn("_copy", F.row_number().over(first_arrival)))
reason = (F.when(F.col("tag_name").isNull() | F.col("ts_utc").isNull(), "null_key")
           .when(F.col("_known_tag").isNull(), "unknown_tag")
           .when(F.col("_copy") > 1, "duplicate_key"))
checked = checked.withColumn("reject_reason", reason).cache()
rejects = (checked.filter(F.col("reject_reason").isNotNull())
    .select("tag_name", "ts_utc", "sent_utc", "value", "source", "stream_run_id", "reject_reason", F.lit(RUN_ID).alias("run_id")))
clean = checked.filter(F.col("reject_reason").isNull()).select(
    "tag_name", "ts_utc", "sent_utc", "value", "source", "stream_run_id", F.lit(True).alias("is_received"))
reject_counts = {r["reject_reason"]: r["count"] for r in rejects.groupBy("reject_reason").count().collect()}
print("rejects:", reject_counts)


In [ ]:
# Expected readings: the local month's hours for the history, the run's sample times for the feed
year, month = (int(x) for x in CFG["month"]["production_month"].split("-"))
tz = ZoneInfo(CFG["month"]["timezone"])
utc = ZoneInfo("UTC")
start_utc = datetime(year, month, 1, tzinfo=tz).astimezone(utc)
end_utc = datetime(year + month // 12, month % 12 + 1, 1, tzinfo=tz).astimezone(utc)
hours = int((end_utc - start_utc).total_seconds() // 3600)
if hours != CFG["month"]["expected_hours"]:
    raise ValueError(f"{hours} hours in the month, expected {CFG['month']['expected_hours']}")
backfill_source = clean.filter(F.col("stream_run_id").isNull()).select("source").distinct()
backfill_times = (spark.range(hours)
    .withColumn("ts_utc", F.timestamp_seconds(F.lit(int(start_utc.timestamp())) + F.col("id") * 3600))
    .select("ts_utc", F.lit(None).cast("string").alias("stream_run_id"))
    .crossJoin(backfill_source))
stream_times = (clean.filter(F.col("stream_run_id").isNotNull()).select("ts_utc", "stream_run_id", "source").distinct())
grid = tags.crossJoin(backfill_times.unionByName(stream_times))
joined = (grid.join(clean.select("tag_name", "ts_utc", "sent_utc", "value", "is_received"), ["tag_name", "ts_utc"], "left")
    .withColumn("is_received", F.coalesce("is_received", F.lit(False)))
    .join(frozen_checked, "tag_code", "left"))


In [ ]:
# Quality flags, the same rules as sensor/quality.py. A missing reading always ends a frozen run.
series = ["tag_name", "source", "stream_run_id"]
in_order = Window.partitionBy(*series).orderBy("ts_utc")
prev = F.lag("value").over(in_order)
new_run = F.col("value").isNull() | prev.isNull() | (F.col("value") != prev)
flagged = (joined
    .withColumn("_new_run", F.when(new_run, 1).otherwise(0))
    .withColumn("_run", F.sum("_new_run").over(in_order))
    .withColumn("_run_len", F.count(F.lit(1)).over(Window.partitionBy(*series, "_run"))))
missing = F.col("value").isNull()
outside_physical = (F.col("value") < F.col("physical_min")) | (F.col("value") > F.col("physical_max"))
outside_normal = (F.col("value") < F.col("normal_min")) | (F.col("value") > F.col("normal_max"))
frozen = (F.col("frozen_check") & (F.col("_run_len") >= Q["frozen_min_points"]) & ~missing
          & ~F.col("value").isin(Q["frozen_ignore_values"]))
late_after = STREAM["late_delay_samples"][0] * STREAM["sample_interval_seconds"]
silver = (flagged
    .withColumn("quality", F.when(missing | outside_physical, "Bad").when(outside_normal | frozen, "Questionable").otherwise("Good"))
    .withColumn("quality_reason", F.when(missing, "missing").when(outside_physical, "outside_physical_range")
                                   .when(outside_normal, "outside_normal_range").when(frozen, "frozen").otherwise(""))
    .withColumn("is_late", F.coalesce(F.unix_timestamp("sent_utc") - F.unix_timestamp("ts_utc") >= late_after, F.lit(False)))
    .withColumn("local_day", F.to_date(F.from_utc_timestamp("ts_utc", CFG["month"]["timezone"])))
    .select("tag_name", "ts_utc", "local_day", "value", "quality", "quality_reason", "source", "stream_run_id", "sent_utc",
            "is_late", "is_received", "well_key", "facility_id", "product", "tag_code", "unit",
            F.lit(RUN_ID).alias("_run_id"))
    .cache())
print("Silver rows:", silver.count())


In [ ]:
# Rule tests. The history has the deliberate bad readings: its counts must equal the injections.
# The feed has none (DEC-034): it must be all Good, with duplicates and late readings as its run manifests say.
with open(L["truth_file"]) as fh:
    injections = json.load(fh)["injections"]
manifests = [json.load(open(p)) for p in glob.glob(L["stream_manifests"])]
loaded_runs = {r[0] for r in clean.select("stream_run_id").distinct().collect() if r[0]}
expected_stream = [m["expected"] for m in manifests if m["run_id"] in loaded_runs]
n_kind = lambda kind: sum(1 for i in injections if i["kind"] == kind)
frozen_points = sum(i["hours"] + 1 for i in injections if i["kind"] == "frozen")
is_history = F.col("stream_run_id").isNull()
history_reasons = {r["quality_reason"]: r["count"] for r in silver.filter(is_history).groupBy("quality_reason").count().collect()}
stream_rows = silver.filter(~is_history)
dup_keys = silver.groupBy("tag_name", "ts_utc").count().filter("count > 1").count()

results = [
    ("history_missing_equals_injected", n_kind("missing"), history_reasons.get("missing", 0)),
    ("history_out_of_range_equals_injected", n_kind("out_of_range"), history_reasons.get("outside_physical_range", 0)),
    ("history_frozen_equals_injected", frozen_points, history_reasons.get("frozen", 0)),
    ("history_outside_normal_is_none", 0, history_reasons.get("outside_normal_range", 0)),
    ("stream_not_good_is_none", 0, stream_rows.filter(F.col("quality") != "Good").count()),
    ("stream_runs_have_manifests", len(loaded_runs), len(expected_stream)),
    ("stream_duplicates_match_manifests", sum(e["duplicates"] for e in expected_stream), reject_counts.get("duplicate_key", 0)),
    ("stream_late_match_manifests", sum(e["late"] for e in expected_stream), stream_rows.filter("is_late").count()),
    ("silver_key_unique", 0, dup_keys),
    ("silver_keys_not_null", 0, silver.filter(F.col("tag_name").isNull() | F.col("ts_utc").isNull() | F.col("well_key").isNull()).count()),
    ("rejects_are_only_duplicates", 0, sum(v for k, v in reject_counts.items() if k != "duplicate_key")),
]
rows = [(RUN_ID, n, int(e), int(a), "PASS" if e == a else "FAIL") for n, e, a in results]
schema = "run_id string, test_name string, expected long, actual long, status string"
spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(L["quality_tests_table"])
for r in rows:
    print(r[4], r[1], "expected", r[2], "actual", r[3])

# Stop before Silver is replaced, so 05_sensor_gold cannot read a table that failed its tests
if rejects.limit(1).count():
    rejects.write.format("delta").mode("append").saveAsTable(L["rejects_table"])
failed = [r for r in rows if r[4] == "FAIL"]
LOG_SCHEMA = ("run_id string, layer string, source string, file string, rows_in long, rows_out long, "
              "status string, message string, started_at timestamp, ended_at timestamp")

def log_run(rows_out, status, msg):
    (spark.createDataFrame([(RUN_ID, "silver", "sensor", L["silver_table"], rows_in, rows_out, status, msg, started, datetime.now())],
                           LOG_SCHEMA).write.format("delta").mode("append").saveAsTable("pipeline_run_log"))

if failed:
    detail = "; ".join(f"{r[1]} (expected {r[2]}, actual {r[3]})" for r in failed)
    log_run(None, "FAILED", detail[:500])
    raise ValueError(f"Sensor Silver rule tests failed for run {RUN_ID}: {detail}. Silver is unchanged.")


In [ ]:
# Write, then log: 05_sensor_gold requires this SUCCESS row
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(L["silver_table"])
rows_out = spark.table(L["silver_table"]).count()
log_run(rows_out, "SUCCESS", "")
print("Silver:", rows_out, "rows | run", RUN_ID)
